# Koddan Kasa · Levha 32 · Robota konuşarak iş yaptırmak (Strands Agents)

Videodaki deney: bir dil modeline **tek bir araç** olarak simüle bir robot kol veriyoruz ve "pick up the red cube" diyoruz.
Sonra ajanın cevabına değil, **ölçüme** bakıyoruz: küp gerçekten kalktı mı?

- Robot: `strands-robots` (AWS Strands Labs), SO-100 kol, MuJoCo simülasyonu. GPU ya da donanım gerekmez.
- Ajan modeli: **A)** Colab içinde Ollama ile yerel `qwen3.5:9b` (videodaki model) ya da **B)** kendi Anthropic API anahtarınla Claude.
- Not: simde parmaklar küpü sürtünmeyle kaldıramıyor; ajan `attach_bodies(mode="weld")` ile küpü çeneye "yapıştırıyor". Bu bir **tutuş yardımı**, fiziksel tutuş değil.
- Bizim sonuçlarımız (Mac, qwen3.5:9b, 6 deneme): 2 denemede küp kalktı, 1 denemede ajan küp masadayken "başardım" dedi.

## 1 · Kurulum

In [ ]:
# strands-robots 0.5.3, MuJoCo 3.17 ile attach_bodies(weld) çağrısında hata veriyor (mjNEQDATA 11 → 12).
# Bu yüzden MuJoCo 3.16.0'a sabitliyoruz.
!pip -q install "strands-robots[sim-mujoco]==0.5.3" "strands-agents[ollama]" "mujoco==3.16.0"
import os
os.environ["MUJOCO_GL"] = "egl"  # Colab'de ekransız çizim

## 2 · Ajanın beyni (birini seç)

**A) Yerel model (videodaki):** Colab'e Ollama kurup `qwen3.5:9b`'yi indiriyoruz (~6,6 GB, GPU çalışma zamanı önerilir).

In [ ]:
USE = "ollama"   # "ollama" ya da "anthropic"

if USE == "ollama":
    !curl -fsSL https://ollama.com/install.sh | sh
    import subprocess, time
    subprocess.Popen(["ollama", "serve"]); time.sleep(5)
    !ollama pull qwen3.5:9b

**B) Claude:** sol menüdeki 🔑 Secrets'a `ANTHROPIC_API_KEY` ekle, `USE = "anthropic"` yap.

In [ ]:
if USE == "ollama":
    from strands.models.ollama import OllamaModel
    llm = OllamaModel(None, model_id="qwen3.5:9b")
else:
    !pip -q install "strands-agents[anthropic]"
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    from strands.models.anthropic import AnthropicModel
    llm = AnthropicModel(model_id="claude-sonnet-5-5", max_tokens=4096)

## 3 · Videodaki 7 satır

In [ ]:
from strands import Agent
from strands_robots import Robot

robot = Robot("so100")   # MuJoCo simülasyonu; mode="real" gerçek kol içindir
robot.add_object(name="red_cube", shape="box", size=[0.05, 0.05, 0.05],
                 position=[0.0, -0.2, 0.025], color=[1.0, 0.0, 0.0])
print(robot.tool_spec["name"], "·", len(robot.tool_spec["inputSchema"]["json"]["properties"]["action"]["enum"]), "eylem")

## 4 · Her araç çağrısını kaydederek koştur

In [ ]:
import json, time, mujoco
from strands.hooks import AfterToolCallEvent

def cube_z():
    m, d = robot.mj_model, robot.mj_data   # attach_bodies modeli yeniden derler: her seferinde taze oku
    return float(d.xpos[mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, "red_cube")][2])

log, t0 = [], time.time()
def after(ev: AfterToolCallEvent):
    text = " ".join(c.get("text", "") for c in (ev.result or {}).get("content", []) if isinstance(c, dict))
    log.append(dict(t=round(time.time() - t0, 1), action=ev.tool_use["input"].get("action"),
                    status=(ev.result or {}).get("status"), cube_z=round(cube_z(), 4), result=text[:200]))
    print(f"[{log[-1]['t']:6.1f}s] {log[-1]['action']:16s} {log[-1]['status']:8s} küp z = {log[-1]['cube_z']*100:.2f} cm")

agent = Agent(model=llm, tools=[robot], callback_handler=None)
agent.hooks.add_callback(AfterToolCallEvent, after)
reply = agent("pick up the red cube")
print("\nAJANIN CEVABI:\n", reply)

## 5 · Cevaba değil, ölçüme bak

In [ ]:
z = cube_z()
print(f"Araç çağrısı: {len(log)}  ·  küp yüksekliği: {z*100:.2f} cm  (masada dururken 2,5 cm)")
print("SONUÇ:", "küp KALKTI" if z > 0.05 else "küp MASADA (ajan ne derse desin)")
print(f"Simülasyon saati: {robot.mj_data.time:.2f} sn  ·  duvar saati: {time.time()-t0:.0f} sn")

## 6 · Son kareye bak

In [ ]:
from PIL import Image
robot.add_camera(name="front", position=[0.3, -0.7, 0.45], target=[0.0, -0.2, 0.15])
res = robot.render(camera_name="front")
print(res["content"][0].get("text", "")[:300])

## Dene

- Aynı istemi 3–5 kez koştur: kaçında küp kalkıyor, kaçında ajan yanlışlıkla "başardım" diyor?
- İstemi Türkçe yaz: `"kırmızı küpü al"`.
- Sistem istemine "kaldırdıktan sonra get_body_state ile küpün yüksekliğini kontrol et" ekle: başarı oranı değişiyor mu?

Kaynaklar: strands-robots (Apache-2.0) https://pypi.org/project/strands-robots/ · README örneği · Hugging Face Robotics Course https://huggingface.co/learn/robotics-course